In [ ]:
import numpy as np
import rioxarray as rioxr
import xarray as xr

In [ ]:
from conus_biomass import dir_info
from conus_biomass.process_inputs.extract_points_from_rasters import (
    calculate_point_buffers,
    calculate_transformed_points,
    extract_from_raster,
)

In [ ]:
from conus_biomass.process_inputs.process_canopy_cover import get_tree_cover_fname

In [ ]:
from conus_biomass.process_outputs.generate_state_csv import get_crs

# Load FIA data

In [ ]:
fia_data = xr.open_mfdataset(
    dir_info.dir_processed + "restructured_FIA/*_FIA_plots_and_PRISM_v7.nc",
    combine="nested",
    concat_dim="plotid",
)
fia_data = fia_data.where(
    fia_data["STATECD"].isin([4, 6, 8, 16, 30, 32, 35, 41, 49, 53, 56]).load(), drop=True
)

In [ ]:
fia_data

# Extract canopy cover data

In [ ]:
dir_source = dir_info.dir_canopy_cover

In [ ]:
year = 2007
fpath = get_tree_cover_fname(year=2005)
da = rioxr.open_rasterio(fpath)

lons = fia_data["lon"].values
lats = fia_data["lat"].values

In [ ]:
points_transformed = calculate_transformed_points(
    lats=fia_data["lat"],
    lons=fia_data["lon"],
    crs_original="EPSG:4326",
    crs_new=da.rio.crs,
)

In [ ]:
buffers = calculate_point_buffers(
    lats=fia_data["lat"],
    lons=fia_data["lon"],
    crs_original="EPSG:4326",
    crs_new=da.rio.crs,
    buffer=1609.34 / 2,
)

In [ ]:
crs, grid_res = get_crs(dir_model_input=dir_info.dir_model_input)

In [ ]:
def get_canopy_cover_data(year: int):
    ds = xr.open_zarr(
        dir_info.dir_model_input + "TREE_CANOPY_COVER/NLCD_TCC_" + str(year) + ".zarr"
    )
    da = ds["LIVE_CANOPY_CVR_PCT"]
    da = da.rio.write_crs(crs)

    # fpath = get_tree_cover_fname(year=year)
    # da = rioxr.open_rasterio(fpath).isel(band=0)

    return da

In [ ]:
canopy_cover_points_NLCD = xr.DataArray(
    np.full((len(fia_data["plotid"]), len(fia_data["year"])), np.nan),
    dims=("plotid", "year"),
    coords={"plotid": fia_data["plotid"], "year": fia_data["year"]},
    name="canopy_cover_NLCD",
)

canopy_cover_buffers_NLCD = xr.DataArray(
    np.full((len(fia_data["plotid"]), len(fia_data["year"])), np.nan),
    dims=("plotid", "year"),
    coords={"plotid": fia_data["plotid"], "year": fia_data["year"]},
    name="canopy_cover_NLCD",
)

In [ ]:
xs = np.array([p.x for p in points_transformed])
ys = np.array([p.y for p in points_transformed])

x_da = xr.DataArray(xs, dims="plotid")
y_da = xr.DataArray(ys, dims="plotid")

years = np.arange(1999, 2024)

In [ ]:
for year in years:
    print(year)
    da_year = get_canopy_cover_data(year=year)

    sampled = da_year.sel(x=x_da, y=y_da, method="nearest")
    canopy_cover_points_NLCD.loc[dict(year=year)] = sampled

In [ ]:
canopy_cover_points_NLCD.name = "extract_canopy_cover_pts"
encoding = {"plotid": {"dtype": "S12"}}
canopy_cover_points_NLCD.to_netcdf("canopy_cover_points.nc", encoding=encoding)

In [ ]:
canopy_cover_buffers_NLCD.name = "extract_canopy_cover_buffer"
encoding = {"plotid": {"dtype": "S12"}}

for year in years:
    print(year)
    da_year = get_canopy_cover_data(year=year)

    sampled_buffer = extract_from_raster(
        buffers=buffers,
        raster_to_extract=da_year,
        # nan_value=da_year.attrs["_FillValue"],
        aggfunc="mean",
    )
    canopy_cover_buffers_NLCD.loc[dict(year=year)] = sampled_buffer

    canopy_cover_buffers_NLCD.sel(year=year).to_netcdf(
        "canopy_cover_buffer_" + str(year) + ".nc", encoding=encoding
    )

# Overwrite canopy cover with NLCD values

In [ ]:
canopy_cover_buffers_NLCD = xr.DataArray(
    np.full((len(fia_data["plotid"]), len(fia_data["year"])), np.nan),
    dims=("plotid", "year"),
    coords={"plotid": fia_data["plotid"], "year": fia_data["year"]},
    name="canopy_cover_NLCD",
)

In [ ]:
years = np.arange(1999, 2024)

In [ ]:
for year in years:
    print(year)
    if year == 2020:
        ds_year = xr.open_dataset("canopy_cover_buffer_" + str(2021) + ".nc")
    else:
        ds_year = xr.open_dataset("canopy_cover_buffer_" + str(year) + ".nc")
    da_year = ds_year["extract_canopy_cover_buffer"]

    # da_year = da_year.assign_coords(plotid=da_year['plotid'].astype('S12'))

    canopy_cover_buffers_NLCD.loc[dict(year=year)] = da_year.values

In [ ]:
fia_data["LIVE_CANOPY_CVR_PCT_FIA"] = fia_data["LIVE_CANOPY_CVR_PCT"].copy()

In [ ]:
fia_data["LIVE_CANOPY_CVR_PCT"] = canopy_cover_buffers_NLCD

In [ ]:
fia_data.to_netcdf(dir_info.dir_processed + "restructured_FIA/Westwide_FIA_plots_and_PRISM_v8.nc")